In [0]:
from functools import reduce
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, TimestampType

In [0]:
tables: list = spark.catalog.listTables("bikes.01_bronze")
trip_tables: list = [table.name for table in tables if "_trips_raw" in table.name]
dfs: list = []
for table in trip_tables:
    df = spark.read.table(f"bikes.01_bronze.{table}")
    dfs.append(df)
merged_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)

In [0]:
merged_df = merged_df.filter("started_at >= '2026-01-01' AND started_at < '2026-08-01'")

In [0]:
merged_df.filter((col("start_station_id").isNotNull()) & (col("end_station_id").isNotNull())).\
    select(
        col("city"),
        col("ride_id"),
        col("rideable_type"),
        col("started_at").cast(TimestampType()),
        col("ended_at").cast(TimestampType()),
        col("start_station_name"),
        col("start_station_id"),
        col("end_station_name"),
        col("end_station_id"),
        col("start_lat").cast(DoubleType()),
        col("start_lng").cast(DoubleType()),
        col("end_lat").cast(DoubleType()),
        col("end_lng").cast(DoubleType()),
        col("member_casual"),
        col("processed_timestamp")
    ).write.mode("overwrite").saveAsTable("bikes.02_silver.trips_cleansed")